# Week 2 — Data Learning Journey
**Name:** Utsav  
**College:** MSIT Delhi (B.Tech IT) + IIT Jodhpur (BS Applied AI & Data Science)  
**Goal:** Become a Data Analyst  

This notebook covers everything learned in Week 2:
- Day 8: SQL JOINs
- Day 9: Pandas merging
- Day 10: Handling missing data
- Days 11-12: First end-to-end project
- Day 13: Project wrap up + GitHub push

---

# Day 8 — SQL JOINs
**Platform:** SQLZoo — The JOIN operation  
**Result:** Completed all 13 questions

## Why JOINs exist
In real databases, data is never all in one table. It is split across multiple tables to avoid repetition.  
A JOIN lets you combine two or more tables into one result using a common column.

Example — a cricket database:
- `matches` table — match id, date, venue, season
- `deliveries` table — match id, batsman, runs, wickets

The two tables connect via `match_id`. A JOIN lets you pull data from both at once.

## Types of JOINs

```
Table A        Table B
  ( A only | BOTH | B only )
```

| JOIN type | What it returns |
|---|---|
| `INNER JOIN` | Only rows that exist in BOTH tables |
| `LEFT JOIN` | Everything from left table + matching rows from right (unmatched = NULL) |
| `RIGHT JOIN` | Opposite of LEFT (rarely used) |
| `FULL JOIN` | Everything from both tables |

**In real work: INNER JOIN and LEFT JOIN cover 95% of cases.**

## INNER JOIN — the most common

### Syntax
```sql
SELECT table1.column, table2.column
FROM table1
INNER JOIN table2
ON table1.common_column = table2.common_column
```

### Example

`students` table:
| student_id | name |
|---|---|
| 1 | Rahul |
| 2 | Priya |
| 3 | Amit |

`marks` table:
| student_id | subject | score |
|---|---|---|
| 1 | Maths | 85 |
| 2 | Maths | 90 |

```sql
-- Get each student name with their score
SELECT students.name, marks.subject, marks.score
FROM students
INNER JOIN marks
ON students.student_id = marks.student_id
```

Result:
| name | subject | score |
|---|---|---|
| Rahul | Maths | 85 |
| Priya | Maths | 90 |

**Amit is missing** — he has no row in marks, so INNER JOIN excludes him entirely.

## LEFT JOIN — keeps everyone from the left table

```sql
SELECT students.name, marks.subject, marks.score
FROM students
LEFT JOIN marks
ON students.student_id = marks.student_id
```

Result:
| name | subject | score |
|---|---|---|
| Rahul | Maths | 85 |
| Priya | Maths | 90 |
| Amit | NULL | NULL |

**Amit appears now but with NULL.**  
To find students who have NOT submitted marks:
```sql
WHERE marks.score IS NULL
```

### Key rule
Whenever you are counting things and some groups might have zero counts, use LEFT JOIN.  
INNER JOIN silently drops those rows and you get wrong results without any error message.

## Table aliases — shortcut for table names

```sql
-- Without alias (verbose)
SELECT students.name, marks.subject
FROM students
INNER JOIN marks ON students.student_id = marks.student_id

-- With alias (clean) — standard in real jobs
SELECT s.name, m.subject
FROM students s
INNER JOIN marks m ON s.student_id = m.student_id
```

## SQLZoo football database — tables used today

- `game` — `id`, `mdate`, `stadium`, `team1`, `team2`
- `goal` — `matchid`, `teamid`, `player`, `gtime`
- `eteam` — `id`, `teamname`, `coach`

How they connect:
- `game` + `goal` → `game.id = goal.matchid`
- `goal` + `eteam` → `goal.teamid = eteam.id`

## TRICKY QUESTION - QUESTION 8
Instead show the name of all players who scored a goal against Germany.
```sql
SELECT DISTINCT(player)
  FROM game JOIN goal ON matchid = id 
    WHERE (teamid!='GER') AND (team1='GER' OR team2='GER')```

## CASE WHEN — conditional logic inside SQL

SQL's version of if/else:
```sql
CASE WHEN teamid = team1 THEN 1 ELSE 0 END
```
Means: if goal scored by team1 → 1, otherwise → 0.

## SUM(CASE WHEN) — conditional aggregation
One of the most common interview patterns. Counts how many times a condition was true per group:

```sql
-- Show scores for each ENG match including 0-0 draws
SELECT mdate,
       team1,
       SUM(CASE WHEN teamid = team1 THEN 1 ELSE 0 END) score1,
       team2,
       SUM(CASE WHEN teamid = team2 THEN 1 ELSE 0 END) score2
FROM game LEFT JOIN goal ON matchid = id
WHERE (team1 = 'ENG' OR team2 = 'ENG')
GROUP BY mdate, matchid, team1, team2
ORDER BY mdate, matchid, team1, team2
```

**Why LEFT JOIN here?**  
Matches where both teams scored 0 have no rows in the goal table.  
INNER JOIN drops those matches. LEFT JOIN keeps them and SUM returns 0 for NULL.

**Mental model:**  
CASE WHEN puts a 1 or 0 in each goal row → SUM adds them up per group = goals per team per match.

## Practice queries — 8 questions written from scratch

```sql
-- 1. Stadium and number of goals scored in each stadium
SELECT g.stadium, COUNT(go.matchid) AS total_goals
FROM game g
LEFT JOIN goal go ON g.id = go.matchid
GROUP BY g.stadium
ORDER BY total_goals DESC

-- 2. Player name and team they scored for
SELECT go.player, go.teamid AS team
FROM goal go
INNER JOIN game g ON go.matchid = g.id

-- 3. Matches where Germany played — date and both team names
SELECT g.mdate, g.team1, g.team2
FROM game g
WHERE g.team1 = 'GER' OR g.team2 = 'GER'

-- 4. Teams that scored more than 5 goals total
SELECT teamid, COUNT(*) AS total_goals
FROM goal
GROUP BY teamid
HAVING COUNT(*) > 5

-- 5. Players who scored more than 2 goals in a single match
SELECT player, matchid, COUNT(*) AS goals
FROM goal
GROUP BY player, matchid
HAVING COUNT(*) > 2

-- 6. Number of goals per match — most goals first
SELECT matchid, COUNT(*) AS total_goals
FROM goal
GROUP BY matchid
ORDER BY total_goals DESC

-- 7. All teams and total goals — including teams that scored 0 (LEFT JOIN)
SELECT e.teamname, COUNT(go.teamid) AS total_goals
FROM eteam e
LEFT JOIN goal go ON e.id = go.teamid
GROUP BY e.teamname
ORDER BY total_goals DESC

-- 8. Player who scored the most goals overall
SELECT player, COUNT(*) AS total_goals
FROM goal
GROUP BY player
ORDER BY total_goals DESC
LIMIT 1
```

## Key patterns to remember from Day 8

| Pattern | When to use |
|---|---|
| `INNER JOIN` | Only want rows with matches in both tables |
| `LEFT JOIN` | Want all rows from left table even with no match |
| `WHERE col IS NULL` | Find rows with no match after a LEFT JOIN |
| `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` | Count how many times a condition is true per group |
| Table aliases | Write cleaner queries with multiple tables |
| Add `matchid` to GROUP BY | Avoid merging two matches on the same date |

## Critical warning
Never write a JOIN without an `ON` clause.  
Without `ON`, SQL creates a cartesian product — every row in table A × every row in table B = garbage.

---


---

# Day 9 — Pandas Merging
**Topic:** Combining DataFrames using merge(), concat(), and join()
**Platform:** Kaggle Pandas Course — Lesson 6 + Jupyter practice
**Result:** Completed lesson, exercises and 6 practice tasks

---

## SQL JOINs vs Pandas — same concept, different syntax

| SQL | Pandas |
|---|---|
| `INNER JOIN` | `merge(how='inner')` |
| `LEFT JOIN` | `merge(how='left')` |
| `RIGHT JOIN` | `merge(how='right')` |
| `FULL OUTER JOIN` | `merge(how='outer')` |

---

## merge() — combining two dataframes on a common column

### INNER JOIN — only matching rows

```python
import pandas as pd

# Table 1 — students
students = pd.DataFrame({
    'student_id': [1, 2, 3],
    'name': ['Rahul', 'Priya', 'Amit']
})

# Table 2 — marks
marks = pd.DataFrame({
    'student_id': [1, 2],
    'subject': ['Maths', 'Maths'],
    'score': [85, 90]
})

# SQL: SELECT * FROM students INNER JOIN marks ON students.student_id = marks.student_id
result = pd.merge(students, marks, on='student_id', how='inner')
print(result)
# Amit is missing — no marks row for him, same as SQL INNER JOIN
```

### LEFT JOIN — keep everyone from left table

```python
# SQL: SELECT * FROM students LEFT JOIN marks ON students.student_id = marks.student_id
result = pd.merge(students, marks, on='student_id', how='left')
print(result)
# Amit appears with NaN — same as SQL NULL
```

### When column names are different in each table

```python
marks2 = pd.DataFrame({
    'id': [1, 2],       # same data, different column name
    'score': [85, 90]
})

# left_on = column name in left table
# right_on = column name in right table
result = pd.merge(students, marks2, left_on='student_id', right_on='id', how='inner')
```

### Merging on multiple columns

```python
# Match on both student_id AND subject
result = pd.merge(df1, df2, on=['student_id', 'subject'], how='inner')
```

### Duplicate column names after merge — _x and _y suffixes

```python
# If both tables have a column called 'name' (other than the join key)
# Pandas automatically adds _x and _y suffixes
result = pd.merge(df1, df2, on='student_id')
# result will have 'name_x' (from df1) and 'name_y' (from df2)

# Fix by renaming
result = result.rename(columns={'name_x': 'student_name', 'name_y': 'teacher_name'})
```

---

## concat() — stacking dataframes vertically

Different from merge. Instead of combining side by side on a common column,
concat stacks dataframes on top of each other — like appending rows.

```python
# Two dataframes with same columns — different batches of students
marks_2023 = pd.DataFrame({
    'name': ['Rahul', 'Priya'],
    'score': [85, 90]
})

marks_2024 = pd.DataFrame({
    'name': ['Amit', 'Sara'],
    'score': [78, 92]
})

# Stack them vertically
all_marks = pd.concat([marks_2023, marks_2024])

# Always reset index after concat — otherwise index keeps original numbers
all_marks = pd.concat([marks_2023, marks_2024], ignore_index=True)
print(all_marks)
```

---

## join() — shortcut for merging on index

join() is basically a shortcut for merge() but joins on the INDEX by default
instead of a column. Less flexible than merge, used occasionally.

```python
# student_id is set as the INDEX, not a column
students = pd.DataFrame({
    'name': ['Rahul', 'Priya', 'Amit']
}, index=[1, 2, 3])

marks = pd.DataFrame({
    'score': [85, 90]
}, index=[1, 2])

# No on= needed — joins on index automatically
result = students.join(marks, how='left')
print(result)
```

---

## merge vs concat vs join — when to use which

| Situation | Use |
|---|---|
| Two tables with a common column | `merge()` |
| Same columns, want to stack rows | `concat()` |
| SQL JOIN equivalent | `merge()` |
| Combining monthly/yearly data files | `concat()` |
| Joining on index | `join()` |

---

## Practice tasks completed in Jupyter

```python
import pandas as pd

students = pd.DataFrame({
    'student_id': [1, 2, 3, 4],
    'name': ['Rahul', 'Priya', 'Amit', 'Sara'],
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Chennai']
})

marks = pd.DataFrame({
    'student_id': [1, 2, 4],
    'subject': ['Maths', 'Maths', 'Maths'],
    'score': [85, 90, 92]
})

attendance = pd.DataFrame({
    'student_id': [1, 2, 3],
    'attendance_pct': [90, 75, 85]
})

# 1. INNER JOIN students and marks
inner = pd.merge(students, marks, on='student_id', how='inner')

# 2. LEFT JOIN — find who has no marks
left = pd.merge(students, marks, on='student_id', how='left')
no_marks = left[left['score'].isnull()]

# 3. Merge all three tables together
merged = pd.merge(students, marks, on='student_id', how='left')
merged = pd.merge(merged, attendance, on='student_id', how='left')

# 4. Filter only Delhi students after full merge
delhi_students = merged[merged['city'] == 'Delhi']

# 5. Concat two dataframes
extra_students = pd.DataFrame({
    'student_id': [5, 6],
    'name': ['Karan', 'Neha'],
    'city': ['Pune', 'Delhi']
})
all_students = pd.concat([students, extra_students], ignore_index=True)

# 6. Check shape after concat
print(all_students.shape)
```

---

**Day 9 key takeaways:**
- `merge()` is the main tool — use it like SQL JOINs
- Always specify `how=` explicitly — don't rely on defaults
- Use `left_on` and `right_on` when column names differ between tables
- `concat()` stacks rows — always use `ignore_index=True` after
- `join()` is a shortcut for index-based merging — use `merge()` in most cases
- After merging, check for `_x` and `_y` columns and rename them

---


---

# Day 10 — Handling Missing Data + Data Cleaning
**Topic:** Cleaning messy real-world data — missing values, wrong types, duplicates
**Platform:** Kaggle Data Cleaning Course Lessons 1 & 2 + Jupyter practice
**Result:** Completed both lessons, exercises and full cleaning practice

---

## Why this matters
In every real dataset — Kaggle, company databases, internship projects — data will be
missing, wrongly formatted, or dirty. 80% of a real analyst's time is spent cleaning
data, not analyzing it. This is the core toolkit for that.

---

## The three main problems in real data

| Problem | Example |
|---|---|
| Missing values | Empty cells, NaN, NULL |
| Wrong data types | A number stored as text |
| Duplicates | Same row appearing twice |

---

## Step 1 — Always run this checklist on every new dataset

```python
import pandas as pd
import numpy as np

# Understand the dataset
df.shape           # how many rows and columns
df.head()          # first 5 rows
df.dtypes          # data types of each column
df.describe()      # statistical summary

# Check for missing values
df.isnull().sum()

# Percentage of missing values per column
df.isnull().sum() / len(df) * 100

# Check for duplicates
df.duplicated().sum()
```

Always run this first on any new dataset before touching anything.

---

## Step 2 — Handling missing values

### Three options

**Option 1 — Drop rows with missing values**
```python
df.dropna()                      # drop any row with at least one missing value
df.dropna(how='all')             # drop only if ALL values in the row are missing
df.dropna(subset=['marks'])      # drop rows where specific column is missing
```
Use when: missing rows are very few (less than 5%) and dropping won't affect analysis.

**Option 2 — Fill with a fixed value**
```python
df.fillna(0)                     # fill all missing values with 0
df['marks'].fillna(0)            # fill specific column with 0
df['city'].fillna('Unknown')     # fill categorical column with a string
```
Use when: missing means zero, or for categorical columns.

**Option 3 — Fill with mean / median / mode**
```python
# Fill with mean (average)
df['marks'].fillna(df['marks'].mean(), inplace=True)

# Fill with median (better when data has outliers)
df['marks'].fillna(df['marks'].median(), inplace=True)

# Fill categorical column with most common value
df['city'].fillna(df['city'].mode()[0], inplace=True)
```
Use when: you want to preserve the statistical distribution of the data.

---

## Decision rule — which option to choose

| Situation | What to do |
|---|---|
| Very few missing rows (less than 5%) | `dropna()` |
| Missing means zero | `fillna(0)` |
| Numeric column, no outliers | `fillna(mean)` |
| Numeric column, has outliers | `fillna(median)` |
| Categorical column | `fillna(mode)` |
| Can't assume anything | leave it and document it |

---

## Step 3 — Fixing wrong data types

Very common — numbers stored as text, dates stored as strings.

```python
df = pd.DataFrame({
    'marks': ['85', '90', '78'],       # stored as strings, should be numbers
    'date': ['2024-01-01', '2024-02-01', '2024-03-01']
})

# Check data types
df.dtypes

# Convert string to number
df['marks'] = df['marks'].astype(int)
# or safer — handles errors
df['marks'] = pd.to_numeric(df['marks'])

# Convert string to datetime
df['date'] = pd.to_datetime(df['date'])

# After converting to datetime, extract parts
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
```

---

## Step 4 — Handling duplicates

```python
df.duplicated()                       # True/False for each row
df.duplicated().sum()                 # count of duplicate rows
df[df.duplicated()]                   # see the actual duplicate rows
df.drop_duplicates(inplace=True)      # remove duplicates, keep first occurrence
df.drop_duplicates(subset=['name'])   # remove based on specific column
```

---

## Step 5 — Cleaning column names

Real datasets often have ugly column names like 'Total Sales (INR)' or 'student_ID'.

```python
# Rename specific columns
df.rename(columns={
    'student_ID': 'student_id',
    'Total Sales (INR)': 'total_sales'
}, inplace=True)

# Make ALL column names lowercase and replace spaces with underscores
# Do this on every dataset — becomes a habit
df.columns = df.columns.str.lower().str.replace(' ', '_')
```

---

## inplace=True — important concept

By default, Pandas operations return a new dataframe without changing the original:
```python
df.fillna(0)               # creates new dataframe, df is unchanged
df.fillna(0, inplace=True) # modifies df directly — this is what you usually want
```

Always use inplace=True when you want to actually change your dataframe.

---

## Full practice — cleaning a messy dataset from scratch

```python
import pandas as pd
import numpy as np

# Intentionally messy dataset
df = pd.DataFrame({
    'Student Name': ['Rahul', 'Priya', None, 'Sara', 'Rahul'],
    'Marks': ['85', None, '78', '92', '85'],
    'City': ['Delhi', 'Mumbai', None, 'Chennai', 'Delhi'],
    'Attendance': [90, 75, None, 60, 90]
})

print("Original dataframe:")
print(df)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

# Step 1 — fix column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Step 2 — fix data types
df['marks'] = pd.to_numeric(df['marks'])

# Step 3 — handle missing values
df['student_name'].fillna('Unknown', inplace=True)
df['marks'].fillna(df['marks'].mean(), inplace=True)
df['city'].fillna(df['city'].mode()[0], inplace=True)
df['attendance'].fillna(df['attendance'].median(), inplace=True)

# Step 4 — remove duplicates
df.drop_duplicates(inplace=True)

# Step 5 — verify everything is clean
print("\nCleaned dataframe:")
print(df)
print("\nMissing values after cleaning:", df.isnull().sum())
print("\nDuplicates after cleaning:", df.duplicated().sum())
```

---

## Lesson 2 — Scaling and Normalization

Different columns have very different ranges — marks 0-100, income 10,000-1,000,000.
ML models get confused by these differences. Scaling brings everything to the same range.

**Normalization (Min-Max Scaling)** — squishes values between 0 and 1:
```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['marks_normalized'] = scaler.fit_transform(df[['marks']])
# 0 = lowest value, 1 = highest value
```

**Standardization (Z-score)** — transforms so mean = 0, std = 1:
```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['marks_standardized'] = scaler.fit_transform(df[['marks']])
```

### How much I need this right now

| Role | How much scaling matters |
|---|---|
| Data Analyst | Rarely — mostly clean and analyze, not build models |
| Data Scientist | Very important — ML models like KNN, SVM are sensitive to scale |

Understood the concept, didn't memorize the math at this stage.

---

## Day 10 key takeaways

- Always run the cleaning checklist first on every new dataset
- Use `isnull().sum()` to see missing values per column
- Choose fillna strategy based on the decision rule table above
- Always fix data types before analyzing — numbers stored as strings cause silent errors
- Clean column names immediately — `str.lower().str.replace(' ', '_')`
- Use `inplace=True` to actually modify your dataframe
- Scaling matters for ML models, not for analysis

---
*(Days 11-12 project work will be added below)*